#### Shape-extraction ceiling

A shape summary obtained from a trajectory fitted by a TimeView-style model can
diverge from the "ground truth" summary for three reasons:

- the spline basis dimension is too low to represent the true trajectory,
- the basis can represent the true trajectory, but the neural network outputs
  poorly fitted spline coefficients,
- the shape extractor mis-reads the shape from the spline it is given.

Before introducing a model, this notebook removes the second of these. Instead
of predicting coefficients from covariates, it obtains the coefficients of a
cubic spline by OLS from noise-free outcomes Y. These are the best possible
coefficients for a given basis, so what remains isolates the shape-extraction
machinery, which we explore as a function of:

- the number of interior knots K,
- the number of observations per individual N.

The accuracy measured for each configuration then constitutes an upper bound on what any trained model can achieve at the same spline resolution. We explore this using Wilkerson tumour trajectories.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

# Data generation
from scripts.shape_uncertainty.simulated_data.data_generator import (
    WilkersonDGP,
    generate_simulated_dataset,
)

# Shape-extraction evaluation
from scripts.shape_uncertainty.experiments.experiments import ExtractionEvaluator
from scripts.shape_uncertainty.shape_extraction.shape_distance import (
    exact_shape_sequence_match, 
    mean_shape_summary_distance,
)


In [2]:
# Data-generating process
T = 1.0
ALPHA_G, ALPHA_D = 0.0, 0.0 # 0 = no unobserved heterogeneity
HYPERPARAMS = {"g0": 2.0, "d0": 180.0, "rho0": 10.0,
               "alpha_g": ALPHA_G, "alpha_d": ALPHA_D}

# Observation design 
NR_OBS = 20 # observations per individual
SIGMA = 0.0 # measurement-noise std
REGULAR = True
SEED = 42

# Dataset sizes
D_TRAIN, D_VAL, D_TEST = 2000, 1000, 1000
D_TOTAL = D_TRAIN + D_VAL + D_TEST

# Spline basis
NR_INTERIOR_KNOTS = 5 # B = K + 4 = 9, matches TimeView

# Shape extraction 
ZETA_REL = 0.0
UPSILON_1_REL, UPSILON_2_REL = 0.0, 0.0
UPSILON_PRUNE_REL, DO_PRUNE = 0.0, False


#### Step 1: Generate a dataset

In [3]:
# Specify the configuration for the data generating process 
dgp = WilkersonDGP(
        T=T, 
        hyperparams=HYPERPARAMS,
        ranges=None, # set to default
)
# Create a dataset from the specified DGP 
dataset = generate_simulated_dataset(
                dgp, 
                D=D_TOTAL,
                N=NR_OBS, 
                sigma=SIGMA, 
                regular=REGULAR,
                include_endpoints=True,
                seed=SEED
)

# Split the dataset into test, validation and training sets 
test, _, _ = dataset.split_test_val_train(D_train=D_TRAIN, D_val=D_VAL, D_test=D_TEST)

#### Step 2: Extraction under ideal conditions

In [4]:
# Create an evaluator for the test dataset
evaluator = ExtractionEvaluator(test)

# Fit a model using OLS and return shape extraction accuracy compared to ground truth 
result = evaluator.evaluate_extraction(
    nr_obs=NR_OBS,
    nr_interior_knots=NR_INTERIOR_KNOTS,
    zeta_rel=ZETA_REL,
    u1_rel=UPSILON_1_REL,
    u2_rel=UPSILON_2_REL,
    upsilon_rel_prune=UPSILON_PRUNE_REL,
    do_prune=DO_PRUNE,
)
true_shapes, predicted_shapes = result["truth"], result["predicted"]
print(f"Exact-sequence-match accuracy: {result['accuracy']:.4f}")
print(f"Mean shape-summary distance:   {result['distance']:.4f}")

# Print table of shape summaries
target_patients = [0, 1, 2, 3, 4, 5]
def _shape_states(summary):
    return [state for state, _ in summary]

df_shapes = pd.DataFrame({
    "patient": target_patients,
    "true_summary": [_shape_states(true_shapes[i]) for i in target_patients],
    "predicted_summary": [_shape_states(predicted_shapes[i]) for i in target_patients],
})
display(df_shapes)

# Print table of transition time points
def _transition_times(summary):
    return [f"{time:.4f}" for _, time in summary]

df_transition_times = pd.DataFrame({
    "patient": target_patients,
    "true_transition_times": [_transition_times(true_shapes[i]) for i in target_patients],
    "predicted_transition_times": [_transition_times(predicted_shapes[i]) for i in target_patients],
})
display(df_transition_times)


Exact-sequence-match accuracy: 1.0000
Mean shape-summary distance:   0.0000


,patient,true_summary,predicted_summary
0,0,"[convex_decreasing, convex_increasing]","[convex_decreasing, convex_increasing]"
1,1,"[convex_decreasing, convex_increasing]","[convex_decreasing, convex_increasing]"
2,2,[convex_increasing],[convex_increasing]
3,3,[convex_decreasing],[convex_decreasing]
4,4,[convex_increasing],[convex_increasing]
5,5,"[convex_decreasing, convex_increasing]","[convex_decreasing, convex_increasing]"


,patient,true_transition_times,predicted_transition_times
0,0,"[0.0000, 0.8619]","[0.0000, 0.8622]"
1,1,"[0.0000, 0.5878]","[0.0000, 0.5877]"
2,2,[0.0000],[0.0000]
3,3,[0.0000],[0.0000]
4,4,[0.0000],[0.0000]
5,5,"[0.0000, 0.3471]","[0.0000, 0.3473]"


#### Step 4: Explore mismatch occurances

In [5]:
# Identify the individuals whose extracted shape sequence differs from the truth
mismatches = [
    {"i": i, "true": true_summary, "predicted": predicted_summary}
    for i, (predicted_summary, true_summary)
    in enumerate(zip(result["predicted"], result["truth"]))
    if not exact_shape_sequence_match(predicted_summary, true_summary)
]

print(f"{len(mismatches)} mismatches out of {len(result['truth'])} individuals")

df_mismatches = pd.DataFrame({
    "patient": [m["i"] for m in mismatches],
    "true_summary": [_shape_states(m["true"]) for m in mismatches],
    "predicted_summary": [_shape_states(m["predicted"]) for m in mismatches],
    "true_transition_times": [_transition_times(m["true"]) for m in mismatches],
    "predicted_transition_times": [_transition_times(m["predicted"]) for m in mismatches],
})
display(df_mismatches)

0 mismatches out of 1000 individuals


,patient,true_summary,predicted_summary,true_transition_times,predicted_transition_times


#### Step 5a: Shape extraction sensitivity to observation count N
- Sweep observation counts with a fixed spline resolution (nr interior knots = 5)

In [6]:
nr_obs_values = [5, 10, 20, 40, 80]
observation_sweep = evaluator.sweep_observations(
    nr_obs_values, 
    NR_INTERIOR_KNOTS,
    zeta_rel=ZETA_REL, 
    fixed_u1_rel=UPSILON_1_REL, 
    fixed_u2_rel=UPSILON_2_REL,
    upsilon_rel_prune=UPSILON_PRUNE_REL, 
    do_prune=DO_PRUNE,
)

df_obs_sweep = pd.DataFrame(observation_sweep)
display(df_obs_sweep)



,nr_obs,accuracy,distance
0,5,0.0,0.372073
1,10,1.0,0.000064
2,20,1.0,0.000045
3,40,1.0,0.000045
4,80,1.0,0.000045


#### Step 5b: Shape extraction sensitivity to basis dimension B
- Sweep knot counts with a fixed observation count (nr obs = 20)

In [7]:
nr_knot_values = [1, 3, 5, 7, 9]
knot_sweep = evaluator.sweep_knots(
    NR_OBS, 
    nr_knot_values,
    zeta_rel=ZETA_REL, 
    fixed_u1_rel=UPSILON_1_REL, 
    fixed_u2_rel=UPSILON_2_REL,
    upsilon_rel_prune=UPSILON_PRUNE_REL, 
    do_prune=DO_PRUNE,
)

df_knot_sweep = pd.DataFrame(knot_sweep)
display(df_knot_sweep)

,nr_interior_knots,accuracy,distance
0,1,0.985,0.001257
1,3,0.996,0.000139
2,5,1.000,0.000045
3,7,1.000,0.000020
4,9,1.000,0.000009
